In [ ]:
#   Libraries

#   import tensorflow as tf

# import detectron2

import numpy
import cv2 as cv
import albumentations as A
import torch
import sklearn
import pycocotools
import matplotlib
import pandas as pd
from pycocotools.coco import COCO
import os
import plotly.express as px
import PIL
import torchvision
import ultralytics
import timm
import lightning
import fiftyone
import labelme
import supervision as sv
import json
import yaml
import seaborn as sns
import plotly.graph_objects as go
import tqdm
import rich
import kornia
import wandb
import tensorboard
import mlflow
import hydra
import omegaconf
import json, os, re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pycocotools.coco import COCO
from tqdm import tqdm
import cv2





matplotlib.use("Agg")          # headless – safe for any env


In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2 as cv

img_dir = os.path.join(c_path, 'train/images')

# ── 1. Build a flat DataFrame of annotations ──────────────────────────────────
ann_records = []
for ann in coco.dataset['annotations']:
    img_info  = coco.imgs[ann['image_id']]
    cat_info  = coco.cats[ann['category_id']]
    x, y, w, h = ann['bbox']
    ann_records.append({
        'ann_id'    : ann['id'],
        'image_id'  : ann['image_id'],
        'file_name' : img_info['file_name'],
        'img_w'     : img_info['width'],
        'img_h'     : img_info['height'],
        'cat_id'    : ann['category_id'],
        'cat_name'  : cat_info['name'],
        'x'         : x,  'y': y,
        'w'         : w,  'h': h,
        'area'      : ann['area'],
        'iscrowd'   : ann['iscrowd'],
    })

df = pd.DataFrame(ann_records)
df['aspect_ratio']   = df['w'] / df['h'].replace(0, np.nan)
df['rel_w']          = df['w'] / df['img_w']
df['rel_h']          = df['h'] / df['img_h']
df['rel_area']       = df['area'] / (df['img_w'] * df['img_h'])

print("── Dataset summary ──────────────────────────────────────────")
print(f"  Images      : {df['image_id'].nunique()}")
print(f"  Categories  : {df['cat_id'].nunique()}")
print(f"  Annotations : {len(df)}  (crowd: {df['iscrowd'].sum()})")
print(f"\n── Bbox stats (pixels) ──────────────────────────────────────")
print(df[['w','h','area']].describe().round(1).to_string())


# ── 2. Annotations per image ──────────────────────────────────────────────────
anns_per_img = df.groupby('image_id').size()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Dataset overview', fontsize=14, fontweight='bold')

axes[0].hist(anns_per_img, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Annotations per image')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Images')
for spine in ['top','right']: axes[0].spines[spine].set_visible(False)

# ── 3. Relative bbox area distribution ────────────────────────────────────────
axes[1].hist(df['rel_area'].clip(0, 0.5), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Relative bbox area (clipped at 0.5)')
axes[1].set_xlabel('bbox area / image area')
axes[1].set_ylabel('Annotations')
for spine in ['top','right']: axes[1].spines[spine].set_visible(False)

# ── 4. Aspect-ratio distribution ──────────────────────────────────────────────
axes[2].hist(df['aspect_ratio'].clip(0, 5), bins=50, color='mediumseagreen', edgecolor='white')
axes[2].axvline(1.0, color='red', linestyle='--', linewidth=1, label='square')
axes[2].set_title('Bbox aspect ratio  w/h  (clipped at 5)')
axes[2].set_xlabel('w / h')
axes[2].set_ylabel('Annotations')
axes[2].legend()
for spine in ['top','right']: axes[2].spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


# ── 5. Top-30 categories by annotation count ──────────────────────────────────
top_cats = (df.groupby('cat_name').size()
              .sort_values(ascending=False)
              .head(30))

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(top_cats.index[::-1], top_cats.values[::-1], color='steelblue')
ax.set_title('Top 30 categories by annotation count', fontweight='bold')
ax.set_xlabel('Annotations')
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


# ── 6. Bbox size scatter (relative w vs h, sampled) ──────────────────────────
sample = df.sample(min(3000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(sample['rel_w'], sample['rel_h'],
                c=sample['rel_area'], cmap='viridis',
                alpha=0.4, s=8)
plt.colorbar(sc, ax=ax, label='rel_area')
ax.set_title('Relative bbox width vs height  (sample)', fontweight='bold')
ax.set_xlabel('w / img_w')
ax.set_ylabel('h / img_h')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


# ── 7. Grid of 6 sample images with bounding boxes ───────────────────────────
COLORS = plt.cm.get_cmap('tab20', df['cat_id'].nunique())
cat_color = {cid: COLORS(i) for i, cid in enumerate(sorted(df['cat_id'].unique()))}

sample_img_ids = df['image_id'].drop_duplicates().sample(6, random_state=7).tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Sample images with ground-truth bboxes', fontsize=14, fontweight='bold')

for ax, img_id in zip(axes.flat, sample_img_ids):
    img_info = coco.imgs[img_id]
    img_path = os.path.join(img_dir, img_info['file_name'])
    bgr = cv.imread(img_path)
    rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns    = coco.loadAnns(ann_ids)

    ax.imshow(rgb)
    for ann in anns:
        x, y, w, h = ann['bbox']
        cid   = ann['category_id']
        color = cat_color[cid]
        rect  = mpatches.Rectangle((x, y), w, h,
                                    linewidth=1.2, edgecolor=color,
                                    facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 3, coco.cats[cid]['name'][:18],
                fontsize=5, color=color, clip_on=True,
                bbox=dict(facecolor='black', alpha=0.35, pad=0, linewidth=0))

    ax.set_title(f"img {img_id}  |  {len(anns)} anns", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()


# ── 8. Annotations-per-category statistics ────────────────────────────────────
cat_stats = (df.groupby('cat_name')
               .agg(n_anns=('ann_id','count'),
                    mean_area=('area','mean'),
                    mean_ar=('aspect_ratio','mean'))
               .sort_values('n_anns', ascending=False))

print("\n── Top 10 categories ────────────────────────────────────────")
print(cat_stats.head(10).round(1).to_string())
print("\n── Bottom 10 categories ─────────────────────────────────────")
print(cat_stats.tail(10).round(1).to_string())